# 04 – Feature Engineering

Dieses Notebook erstellt domänenspezifische Features für das ML-Modell.

**Feature-Kategorien:**
- Kalendarisch: Wochentag, Wochenende, Heizperiode
- Thermodynamisch: Temperaturdelta, thermische Trägheit
- Preisbezogen: Lag-Features, gleitende Durchschnitte
- Gebäudespezifisch: Renovierungsindex, Gebäudetyp-Dummies
- Solar: Potenzial aus PV-Größe × Sonnenscheindauer

**Selektion:** Korrelations-Check (Schwellenwert 0.90) + SHAP-basierte Importance

In [1]:
%pip install pandera polars xgboost matplotlib seaborn lightgbm optuna

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   ------------------- -------------------- 1.0/2.1 MB 2.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.8/2.1 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.4 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 2.0.2

   ---------------------------------------- 0/7 [numpy]
   ---------------------------------------- 0/7 [numpy]
   ---------------------------------------- 0/7 [numpy]
    Uninstalling numpy-2.0.2:
   ---------------------------------------- 0/7 [numpy]
   ---------------------------------------- 0/7 [numpy]
   ---------------------------------------- 0/7 [numpy]
   ---------------------------------------- 0/7 [numpy]
   -

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pycaret 3.3.2 requires joblib<1.4,>=1.2.0, but you have joblib 1.5.3 which is incompatible.
pycaret 3.3.2 requires pandas<2.2.0, but you have pandas 3.0.1 which is incompatible.
pycaret 3.3.2 requires scipy<=1.11.4,>=1.6.1, but you have scipy 1.17.1 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
sktime 0.26.0 requires pandas<2.2.0,>=1.1, but you have pandas 3.0.1 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import sys
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import target_col, feature_cols, feature_cols_cleaned
from data_cleaner import check_amount_nulls
from features import (
    add_weekdays, add_weekend, heating_season, renovation_index,
    heating_amount, add_price_features, solar_potentials,
    add_thermal_dynamics, add_cyclic_features, add_building_type_dummies,
    add_heatpump, add_pv
)
import os
from utilis import train_test_split, correlated_features_drop, feature_importance_analyse, compare_models, tune_lightgbm

## 4.1 Daten laden

In [3]:
data_combined_cleaned = pl.read_parquet(
    str(root_dir / '02_data' / 'processed' / 'combined_data_cleaned.parquet')
)

## 4.2 Features erstellen

In [4]:
df_ml = (
    data_combined_cleaned
    .pipe(add_weekdays)             # Wochentag (1-7)
    .pipe(add_weekend)              # Wochenend-Dummy
    .pipe(heating_season)           # Heizperiode (Okt-Mär)
    .pipe(renovation_index)         # Summe Renovierungs-Dummies
    .pipe(heating_amount)           # Fläche × Bewohner
    .pipe(add_price_features)       # Lag-Features & gleitende Durchschnitte
    .pipe(solar_potentials)         # PV-Größe × Sonnenscheindauer
    .pipe(add_thermal_dynamics)     # EMA 3d + Temperaturdelta
    .pipe(add_cyclic_features)      # Sinus/Kosinus für Monate
    .pipe(add_building_type_dummies)# One-Hot-Encoding Gebäudetyp
    .pipe(add_heatpump) # Wärmepumpen Info
    .pipe(add_pv)# PV Info
    .select(['date', target_col] + feature_cols)
)

print(f'Features: {len(feature_cols)} | Zielvariable: {target_col}')
check_amount_nulls(df_ml)

Features: 21 | Zielvariable: kwh_received_total

---------------------------------------------
TOP 20 SPALTEN MIT MEISTEN NULLS
Datensatz: 937,456 Zeilen | 23 Spalten
Spalten mit Nulls: 1
---------------------------------------------
shape: (1, 3)
┌─────────────────────────┬────────────┬───────┐
│ spalte                  ┆ null_count ┆ pct   │
│ ---                     ┆ ---        ┆ ---   │
│ str                     ┆ u32        ┆ f64   │
╞═════════════════════════╪════════════╪═══════╡
│ solar_thermal_potential ┆ 101171     ┆ 10.79 │
└─────────────────────────┴────────────┴───────┘


spalte,null_count,pct
str,u32,f64
"""solar_thermal_potential""",101171,10.79


In [5]:
df_ml

date,kwh_received_total,kwh_received_heatpump,kwh_returned_total,temperature_avg_daily,building_type_multi_family_house,building_type_other,building_type_single_family_house,swissix_base,weekday,is_weekend,is_heating_season,renovation_score,heating_amount,price_lag_30d,price_relative_to_month,solar_thermal_potential,temp_inertia_ema_3d,temp_delta_1d,month_sin,month_cos,is_heatpump,is_pv
date,f64,f64,f64,f64,i8,i8,i8,f64,i8,i32,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8
2019-03-03,18.33,0.0,8.64,8.4,0,0,0,22.79,7,1,1,0,0.0,22.79,1.022139,0.0,8.4,0.0,1.0,6.1232e-17,0,1
2019-03-04,15.03,0.0,9.04,7.8,0,0,0,37.938333,1,0,1,0,0.0,22.79,1.022139,0.0,8.0,-0.6,1.0,6.1232e-17,0,1
2019-03-05,16.69,0.0,4.57,7.4,0,0,0,37.9225,2,0,1,0,0.0,22.79,1.022139,0.0,7.657143,-0.4,1.0,6.1232e-17,0,1
2019-03-06,29.52,0.0,15.27,8.4,0,0,0,40.97875,3,0,1,0,0.0,22.79,1.022139,0.0,8.053333,1.0,1.0,6.1232e-17,0,1
2019-03-07,16.81,0.0,14.43,9.0,0,0,0,34.819167,4,0,1,0,0.0,22.79,1.022139,0.0,8.541935,0.6,1.0,6.1232e-17,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-03-17,10.14,0.0,5.7,7.7,0,0,0,60.337917,7,1,1,0,0.0,65.44125,0.856396,null,7.699999,0.0,1.0,6.1232e-17,0,1
2024-03-18,12.9,0.0,0.65,7.7,0,0,0,81.739583,1,0,1,0,0.0,67.554583,1.152423,null,7.699999,0.0,1.0,6.1232e-17,0,1
2024-03-19,13.83,0.0,0.16,7.7,0,0,0,84.41875,2,0,1,0,0.0,56.08875,1.174558,null,7.7,0.0,1.0,6.1232e-17,0,1


## 4.3 Train-Test-Split & Korrelations-Check

In [5]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Chronologischer 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    df_ml, feature_cols=feature_cols, target_col=target_col
)

# Korrelierte Features entfernen (Schwellenwert 0.90)
X_train_cleaned, X_test_cleaned, _ = correlated_features_drop(
    X_train, X_test, feautre_cols=feature_cols
)
print(f'Features nach Korrelations-Check: {X_train_cleaned.shape[1]}')

Entfernte redundante Features: ['temp_inertia_ema_3d', 'is_heatpump']
Verbleibende Features (19): ['kwh_received_heatpump', 'kwh_returned_total', 'temperature_avg_daily', 'building_type_multi_family_house', 'building_type_other', 'building_type_single_family_house', 'swissix_base', 'weekday', 'is_weekend', 'is_heating_season', 'renovation_score', 'heating_amount', 'price_lag_30d', 'price_relative_to_month', 'solar_thermal_potential', 'temp_delta_1d', 'month_sin', 'month_cos', 'is_pv']
Features nach Korrelations-Check: 19


## 4.4 SHAP Feature-Importance-Analyse

In [7]:
# RandomForest als Basis für SHAP (Sample für Speed)
sample_idx = np.random.choice(len(X_train_cleaned), 20_000, replace=False)
rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_cleaned.iloc[sample_idx], y_train[sample_idx])

feature_importance_analyse(
    model=rf,
    X_test=X_test_cleaned,
    output_dir=str(root_dir / '05_plots'),
    filename='shap_plot.png'
)

Beide Plots wurden in ..\05_plots gespeichert.


## 4.5 Reduziertes Feature-Set

Auf Basis der SHAP-Analyse werden folgende Features entfernt:
- `heating_amount`: Kaum Informationsgehalt durch hohe Missings
- `building_type_*`: SHAP ≈ 0
- `solar_thermal_potential`: Keine Varianz
- `is_weekend`: Minimaler Effekt
- `renovation_score`: Kein Einfluss
- `is_heating_season`: Redundant zu `temperature_avg_daily`
- `is_pv`: Sehr geringer Einfluss

In [10]:
X_train_red = X_train[feature_cols_cleaned]
X_test_red  = X_test[feature_cols_cleaned]

X_train_final, X_test_final, final_features = correlated_features_drop(
    X_train_red, X_test_red, feautre_cols=feature_cols_cleaned
)
print(f'Finales Feature-Set ({len(final_features)}): {final_features}')

Entfernte redundante Features: []
Verbleibende Features (10): ['temperature_avg_daily', 'kwh_returned_total', 'temp_delta_1d', 'kwh_received_heatpump', 'price_lag_30d', 'price_relative_to_month', 'swissix_base', 'weekday', 'month_sin', 'month_cos']
Finales Feature-Set (10): ['temperature_avg_daily', 'kwh_returned_total', 'temp_delta_1d', 'kwh_received_heatpump', 'price_lag_30d', 'price_relative_to_month', 'swissix_base', 'weekday', 'month_sin', 'month_cos']


## 4.6 Modellvergleich

In [11]:
comparison = compare_models(X_train_final, X_test_final, y_train, y_test)
print(comparison)

--- Benchmark gestartet (Sample-Größe: 5000) ---
  -> Teste Ridge (Linear)... Fertig (0.01s)
  -> Teste XGBoost (Fast)... Fertig (0.14s)
  -> Teste LightGBM... Fertig (0.11s)
  -> Teste RandomForest (Limited)... Fertig (0.28s)
                    Model  MAE_kWh  R2_Score  Duration_sec
3  RandomForest (Limited)   12.296     0.292          0.28
2                LightGBM   12.621     0.268          0.11
0          Ridge (Linear)   12.758     0.285          0.01
1          XGBoost (Fast)   13.440     0.180          0.14


Der Benchmark zeigt, dass RandomForest mit einem MAE von 12,15 kWh und einem 
𝑅
2
R
2
 von 0,308 die beste Prognosegüte erzielt. LightGBM erreicht mit einem MAE von 12,30 kWh und einem 
𝑅
2
R
2
 von 0,293 ein nahezu gleichwertiges Ergebnis, während Ridge und XGBoost deutlich schlechter abschneiden. Aufgrund der vergleichbaren Performance sowie der besseren Skalierbarkeit und Interpretierbarkeit wurde LightGBM als finales Modell ausgewählt.

## 4.7 LightGBM Tuning – Standard

In [12]:
lgbm_model_std, study_std = tune_lightgbm(
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    final_features=final_features,
    output_dir=str(root_dir / '05_plots'),
    n_trials=50, log_transform=False
)

🔍 Starte Optuna Tuning...


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Bestes MAE (CV): 11.624 kWh
✅ Beste Parameter: {'n_estimators': 767, 'learning_rate': 0.09100588514646762, 'num_leaves': 49, 'max_depth': 11, 'min_child_samples': 58, 'subsample': 0.8697786270588249, 'colsample_bytree': 0.964523907905527, 'reg_alpha': 6.532073795695254e-08, 'reg_lambda': 0.04013956474803042}
📊 Lernkurven gespeichert: ..\05_plots\lgbm_learning_curve_std.png

📈 Test-Ergebnisse:
   MAE:  12.031 kWh
   R²:   0.340
💾 Modell gespeichert: ..\05_plots\lgbm_final_model_std.pkl


## 4.8  LightGBM Tuning – Log-Transformation

In [13]:
lgbm_model_log, study_log = tune_lightgbm(
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    final_features=final_features,
    output_dir=str(root_dir / '05_plots'),
    n_trials=50, log_transform=True
)

🔍 Starte Optuna Tuning...


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Bestes MAE (CV): 11.212 kWh
✅ Beste Parameter: {'n_estimators': 636, 'learning_rate': 0.05266956940778808, 'num_leaves': 105, 'max_depth': 3, 'min_child_samples': 63, 'subsample': 0.8616795275564958, 'colsample_bytree': 0.6614094333195217, 'reg_alpha': 0.0009330581215001699, 'reg_lambda': 3.9525313789747304e-05}
📊 Lernkurven gespeichert: ..\05_plots\lgbm_learning_curve_log.png

📈 Test-Ergebnisse:
   MAE:  11.373 kWh
   R²:   0.318
💾 Modell gespeichert: ..\05_plots\lgbm_final_model_log.pkl


## 4.9 Modellvergleich Standard vs. Log

In [15]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred_std = lgbm_model_std.predict(X_test_final)
y_pred_log = np.expm1(lgbm_model_log.predict(X_test_final))

mae_std = mean_absolute_error(y_test, y_pred_std)
mae_log = mean_absolute_error(y_test, y_pred_log)

print(f'Standard – MAE: {mae_std:.3f} kWh | R²: {r2_score(y_test, y_pred_std):.3f}')
print(f'Log      – MAE: {mae_log:.3f} kWh | R²: {r2_score(y_test, y_pred_log):.3f}')

if mae_log < mae_std:
    print('✅ Log-Transformation ist besser')
    lgbm_model = lgbm_model_log
    y_pred = y_pred_log
else:
    print('✅ Standard ist besser')
    lgbm_model = lgbm_model_std
    y_pred = y_pred_std

Standard – MAE: 12.031 kWh | R²: 0.340
Log      – MAE: 11.373 kWh | R²: 0.318
✅ Log-Transformation ist besser


In [18]:
from utilis import (
 plot_final_prediction,
    plot_business_drivers, validate_model_assumptions
)

validate_model_assumptions(
    model=lgbm_model,
    X_test=X_test_final,
    y_test=y_test,
    output_dir=os.path.join(root_dir, "05_plots"),
    log_transform=(mae_log < mae_std) 
)



   Mean Residuum:      3.037 kWh  (sollte ~0 sein)
   Std Residuum:       17.055 kWh
   Max Unterschätzung: 195.139 kWh
   Max Überschätzung:  -76.723 kWh
📊 Validierungsplot gespeichert: ..\05_plots\model_validation.png


array([ -0.49112261,  -0.62027469,  -8.50324268, ...,  -2.93764064,
         5.90466115, -10.27764064])

In [19]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate

tscv = TimeSeriesSplit(n_splits=3)
scores = cross_validate(
    lgbm_model, X_train_final, y_train,
    cv=tscv,
    scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"},
    return_train_score=False
)

for i, (mae, r2) in enumerate(zip(-scores["test_MAE"], scores["test_R2"])):
    print(f"Fold {i+1}: MAE={mae:.3f} kWh | R²={r2:.3f}")

print(f"\nCV MAE:  {-scores['test_MAE'].mean():.3f} ± {scores['test_MAE'].std():.3f} kWh")
print(f"CV R²:   {scores['test_R2'].mean():.3f} ± {scores['test_R2'].std():.3f}")


Fold 1: MAE=17.101 kWh | R²=0.166
Fold 2: MAE=10.456 kWh | R²=0.335
Fold 3: MAE=10.909 kWh | R²=0.424

CV MAE:  12.822 ± 3.031 kWh
CV R²:   0.308 ± 0.107


In [21]:

plot_final_prediction(
    y_test=y_test,
    y_pred=y_pred,
    output_dir=os.path.join(root_dir, "05_plots")
)

plot_business_drivers(
    model=lgbm_model,
    X_sample=X_test_final.sample(2000, random_state=42),
    output_dir=os.path.join(root_dir, "05_plots")
)


📊 Prediction Plot gespeichert: ..\05_plots\final_prediction.png
📊 Business Drivers Plot gespeichert: ..\05_plots\business_drivers.png
